# Plot phenotype cross-products -- per trait, coloured by covariate-set model

Reads the merged `<phenotype>__<transform>__<covariate_set>_merged.full.tsv` files that
`grm_shard_processing.ipynb` / `grm_shard_accumulate_parallel.ipynb` already produced and
uploaded to `CROSSPRODUCTS_BUCKET_GS`. For each phenotype (one plot per `transform`), draws
mean cross-product vs. GRM relatedness bin, one coloured line per covariate-set model (`base`,
`base_pcs`, `base_pcs_zip3`, `base_pcs_zip3_ses` -- whatever's actually present), in `ggplot2`.

Every plot is both shown inline and uploaded back up to
`{BUCKET_DIR_GS}/data/04_process_shards/crossproducts/plots/` -- doesn't touch the `.tsv`
inputs sitting alongside it in the same bucket directory.

## Setup

In [ ]:
required_pkgs <- c("dplyr", "readr", "tidyr", "stringr", "ggplot2", "patchwork")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

library(dplyr)
library(readr)
library(tidyr)
library(stringr)
library(ggplot2)
library(patchwork)

theme_set(theme_bw(base_size = 12))
options(repr.plot.width = 8, repr.plot.height = 6)

## Paths

`CROSSPRODUCTS_DIR` is the local (FUSE-mounted) view of the same bucket directory both
processing notebooks write `*_merged.full.tsv` into -- read locally (no `gsutil`/`gcloud`
download needed), only the *plots* get pushed back up via `gcloud storage cp` at the end.
`SLOPE_FIT_RANGE` mirrors `grm_shard_processing.ipynb`'s convention -- not used for a slope
line here (kept simple/ggplot-native via `geom_smooth`), just documented for reference.

In [ ]:
WORKSPACE_BUCKET <- path.expand("~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")
WORKSPACE_BUCKET_GS <- "gs://shared-env-pilot-wb-fulgent-almond-9474"

CROSSPRODUCTS_DIR <- file.path(WORKSPACE_BUCKET, "data", "04_process_shards", "crossproducts")
CROSSPRODUCTS_BUCKET_GS <- paste0(WORKSPACE_BUCKET_GS, "/data/04_process_shards/crossproducts")
PLOTS_BUCKET_GS <- paste0(CROSSPRODUCTS_BUCKET_GS, "/plots")

LOCAL_PLOTS_DIR <- path.expand("~/grm_pheno_cov/plots_ggplot")
dir.create(LOCAL_PLOTS_DIR, recursive = TRUE, showWarnings = FALSE)

stopifnot(dir.exists(CROSSPRODUCTS_DIR))
sprintf("Reading merged crossproducts from: %s", CROSSPRODUCTS_DIR)

## Discover + load merged files

Filenames are `<phenotype>__<transform>__<covariate_set>_merged.full.tsv` (the `tag` +
`_merged.full.tsv` suffix both processing notebooks write). Parsed the same
`rsplit("__", 2)`-equivalent way `grm_shard_processing.ipynb` parses `.pheno` filenames, so a
new covariate-set or transform name doesn't need any code change here.

In [ ]:
merged_files <- list.files(CROSSPRODUCTS_DIR, pattern = "_merged\\.full\\.tsv$", full.names = TRUE)
merged_files <- merged_files[!str_detect(basename(merged_files), "cheat")]   # exclude earlier cheat/test files
stopifnot(length(merged_files) > 0)

parse_tag <- function(path) {
  stem <- str_remove(basename(path), "_merged\\.full\\.tsv$")
  parts <- str_split(stem, "__", n = 3)[[1]]
  if (length(parts) != 3) return(NULL)
  list(phenotype_name = parts[1], transform = parts[2], covariate_set = parts[3])
}

all_data <- bind_rows(lapply(merged_files, function(path) {
  tag <- parse_tag(path)
  if (is.null(tag)) {
    message(sprintf("Skipping unparseable filename: %s", basename(path)))
    return(NULL)
  }
  read_tsv(path, show_col_types = FALSE) %>%
    mutate(phenotype_name = tag$phenotype_name, transform = tag$transform,
           covariate_set = tag$covariate_set, .before = 1)
}))

phenotypes <- sort(unique(all_data$phenotype_name))
covariate_sets <- sort(unique(all_data$covariate_set))
sprintf("%d phenotypes, %d covariate-set models, %d merged files loaded",
        length(phenotypes), length(covariate_sets), length(merged_files))

## Plot + save + upload, per phenotype

One plot per `(phenotype, transform)`, `covariate_set` (the covariate-set "model") as colour --
error bars from `jk_se`, only bins with `full_n > 0`.

Two weighted-regression fits per covariate-set model, drawn as dotted/dashed lines extrapolated
across the *whole* plotted x-range (fit only within the restricted range each is named for):

- **unrelated fit** -- bins with relatedness in `[-0.02, 0.02]`
- **sib fit** -- bins with relatedness in `[0.45, 0.55]`

Both use a proper inverse-variance-weighted least-squares fit (`lm(weights = 1/jk_se^2)`) --
`jk_se` is a standard error, so the WLS weight is `1/se^2`, *not* `1/se`. (`1/se` is
`numpy.polyfit`'s own convention for its `w` argument, which squares the weight internally --
different libraries, easy to carry the wrong one over by habit.)

A small coefficient panel to the right of each main plot shows the two fitted slopes (with 95%
CI) per covariate-set model, so the sib/unrelated slope estimates are readable at a glance
without eyeballing line steepness. Both panels are combined with `patchwork` into one figure,
saved locally, then uploaded to `PLOTS_BUCKET_GS` with `gcloud storage cp` right after each plot
(not batched at the end), same upload-as-you-go convention as the accumulate step itself.

In [ ]:
covset_colors <- setNames(
  scales::viridis_pal()(length(covariate_sets)),
  covariate_sets
)

FIT_RANGES <- list(
  unrelated = c(-0.02, 0.02),
  sib       = c(0.45, 0.55)
)
FIT_LINETYPES <- c(unrelated = "dashed", sib = "dotted")
FIT_LABELS <- c(unrelated = "unrelated fit (-0.02 to 0.02)", sib = "sib fit (0.45 to 0.55)")

fit_weighted_slope <- function(d, range) {
  d_fit <- d %>%
    filter(bin_midpoint >= range[1], bin_midpoint <= range[2],
           full_n > 0, !is.na(jk_se), jk_se > 0)
  if (nrow(d_fit) < 2) return(NULL)
  # proper inverse-variance WLS -- weight = 1/se^2 (lm()'s `weights` arg is applied directly to
  # squared residuals, unlike numpy.polyfit's `w`, which squares the weight internally -- the two
  # conventions are easy to conflate but aren't the same)
  fit <- lm(full_mean ~ bin_midpoint, data = d_fit, weights = 1 / jk_se^2)
  co <- summary(fit)$coefficients
  tibble(
    intercept = co["(Intercept)", "Estimate"],
    slope = co["bin_midpoint", "Estimate"],
    slope_se = co["bin_midpoint", "Std. Error"],
    n_bins = nrow(d_fit)
  )
}

plot_phenotype_transform <- function(phenotype_name, transform, d) {
  d <- d %>% filter(full_n > 0)
  d$covariate_set <- factor(d$covariate_set, levels = covariate_sets)
  x_range <- range(d$bin_midpoint)

  fit_lines <- bind_rows(lapply(covariate_sets, function(cs) {
    d_cs <- d %>% filter(covariate_set == cs)
    bind_rows(lapply(names(FIT_RANGES), function(fit_type) {
      fit <- fit_weighted_slope(d_cs, FIT_RANGES[[fit_type]])
      if (is.null(fit)) {
        message(sprintf("  not enough bins in %s range for %s/%s/%s slope fit",
                         fit_type, phenotype_name, transform, cs))
        return(NULL)
      }
      fit %>% mutate(covariate_set = cs, fit_type = fit_type, .before = 1)
    }))
  }))

  p <- ggplot(d, aes(x = bin_midpoint, y = full_mean, color = covariate_set)) +
    geom_hline(yintercept = 0, color = "grey70", linewidth = 0.4) +
    geom_vline(xintercept = 0, color = "grey70", linewidth = 0.4) +
    geom_errorbar(aes(ymin = full_mean - jk_se, ymax = full_mean + jk_se), width = 0, alpha = 0.6) +
    geom_point(size = 1.3) +
    geom_line(linewidth = 0.4, alpha = 0.7) +
    scale_color_manual(values = covset_colors)

  if (nrow(fit_lines) > 0) {
    # fit within the restricted range, but draw the line out across the whole plotted x-range
    p <- p +
      geom_segment(
        data = fit_lines,
        aes(x = x_range[1], xend = x_range[2],
            y = intercept + slope * x_range[1], yend = intercept + slope * x_range[2],
            color = covariate_set, linetype = fit_type),
        linewidth = 0.6, inherit.aes = FALSE
      ) +
      scale_linetype_manual(values = FIT_LINETYPES, labels = FIT_LABELS)
  }

  p <- p +
    labs(
      x = "GRM relatedness (bin midpoint)",
      y = "Mean phenotype cross-product (covariance)",
      color = "Covariate-set model",
      linetype = "Weighted regression",
      title = sprintf("%s (%s)", phenotype_name, transform)
    ) +
    theme(plot.title = element_text(hjust = 0.5), legend.position = "top", legend.box = "vertical")

  # small coefficient side panel -- fitted slope +/- 95% CI, per model x fit range
  if (nrow(fit_lines) > 0) {
    coef_data <- fit_lines %>%
      arrange(covariate_set, fit_type) %>%
      mutate(label = paste0(covariate_set, " (", fit_type, ")"))
    coef_data$label <- factor(coef_data$label, levels = rev(unique(coef_data$label)))

    p_coef <- ggplot(coef_data, aes(x = slope, y = label, color = covariate_set)) +
      geom_vline(xintercept = 0, color = "grey70", linewidth = 0.4) +
      geom_segment(aes(x = slope - 1.96 * slope_se, xend = slope + 1.96 * slope_se, yend = label),
                   linewidth = 0.8) +
      geom_point(size = 1.8) +
      scale_color_manual(values = covset_colors, guide = "none") +
      labs(x = "Slope (95% CI)", y = NULL, title = "Coefficients") +
      theme(plot.title = element_text(hjust = 0.5, size = 10), axis.text.y = element_text(size = 7))
  } else {
    p_coef <- plot_spacer()
  }

  combined <- p + p_coef + plot_layout(widths = c(3, 1))

  out_file <- sprintf("%s__%s.png", phenotype_name, transform)
  local_path <- file.path(LOCAL_PLOTS_DIR, out_file)
  ggsave(local_path, combined, width = 11, height = 6, dpi = 150)

  print(combined)   # inline, right here -- visible before the next phenotype starts

  system2("gcloud", c("storage", "cp", shQuote(local_path), shQuote(paste0(PLOTS_BUCKET_GS, "/"))))
  local_path
}

saved_paths <- c()
for (i in seq_along(phenotypes)) {
  phenotype_name <- phenotypes[i]
  message(sprintf("=== [%d/%d] %s ===", i, length(phenotypes), phenotype_name))
  pheno_data <- all_data %>% filter(phenotype_name == !!phenotype_name)
  for (transform in sort(unique(pheno_data$transform))) {
    d <- pheno_data %>% filter(transform == !!transform)
    saved_paths <- c(saved_paths, plot_phenotype_transform(phenotype_name, transform, d))
  }
}

sprintf("done -- %d plots saved locally to %s and uploaded to %s",
        length(saved_paths), LOCAL_PLOTS_DIR, PLOTS_BUCKET_GS)

## Copying already-existing outputs to the bucket

Pushes up everything important that might only exist locally so far: the merged
`*_merged.full.tsv` crossproduct results (`grm_shard_processing.ipynb`'s `WORK_DIR`,
`~/grm_pheno_cov`, doesn't upload these itself -- only `grm_shard_accumulate_parallel.ipynb`
does) and the plot PNGs from either plotting notebook. Skips per-shard `*.acc.tsv`/aligned
`.pheno` files on purpose -- those are intermediate working files, not results worth keeping
in the bucket. Safe to rerun -- `gcloud storage cp` just re-uploads the same bytes if a file's
already there.

In [ ]:
local_merged_tsvs <- list.files(path.expand("~/grm_pheno_cov"), pattern = "_merged\\.full\\.tsv$", full.names = TRUE)
if (length(local_merged_tsvs) > 0) {
  message(sprintf("Copying %d merged .full.tsv files to %s", length(local_merged_tsvs), CROSSPRODUCTS_BUCKET_GS))
  system2("gcloud", c("storage", "cp", shQuote(local_merged_tsvs), shQuote(paste0(CROSSPRODUCTS_BUCKET_GS, "/"))))
} else {
  message("No local merged .full.tsv files found, skipping")
}

for (dir in c("~/grm_pheno_cov/plots", "~/grm_pheno_cov/plots_ggplot")) {
  dir <- path.expand(dir)
  pngs <- list.files(dir, pattern = "\\.png$", full.names = TRUE)
  if (length(pngs) == 0) {
    message(sprintf("No PNGs found in %s, skipping", dir))
    next
  }
  message(sprintf("Copying %d PNGs from %s to %s", length(pngs), dir, PLOTS_BUCKET_GS))
  system2("gcloud", c("storage", "cp", shQuote(pngs), shQuote(paste0(PLOTS_BUCKET_GS, "/"))))
}